Assignement 2

Group - 19

Sarvadnya, Ved, Siddharth

**For all analyses, the training dataset path and the results directory path must be specified separately. This explicit separation ensures clarity, prevents accidental overwrites, and makes each experiment fully reproducible. The directory structure for Fine‑Tuning_Strategies is fixed, as it contains the model weights that are subsequently used for probing different layers. Maintaining a fixed structure guarantees consistency, simplifies reuse of weights, and supports systematic layer‑wise analysis. In Sections 4.3, 4.4, and 4.5, we rely exclusively on fully fine‑tuned models to ensure consistency across experiments. This choice provides a stable baseline, eliminates variability from partial training, and enables meaningful comparisons of representation quality at different depths.**

Mounting and unzipping the data files and also looking how the data looks

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip /content/drive/MyDrive/train_data.zip -d /content/

In [ ]:
!ls /content/train_data

# **4.1 Linear Probe Transfer**
Give the path to your data directory in

DATA_DIR = "path to training dataset"

RESULT_DIR = "path to store results of linear proble transfer strategies"
```bash
|-LinearProbeTransfer
    |-results
        |-densenet121
            |-log
                |-training log
            |-plots (accuracy_curve.png, confusion_matrix.png, feature_pcs.png, feature_tsne.png)                
            |-metrics.txt
            |-model.pth
        |-efficientnet_b0
        |-resnet-50
    |-LinearProbleTransfer.py
    |-evaluate.py
```


In [ ]:
import os
import random
import torch
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.metrics import confusion_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

############################################
# CONSTANT SEED
############################################

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


############################################
# CONFIGURATION
############################################
DATA_DIR = "/content/train_data"
RESULT_DIR = "/content/drive/MyDrive/Linear_Probe"
os.makedirs(RESULT_DIR, exist_ok=True)

MODELS = [
    "resnet50",
    "densenet121",
    "efficientnet_b0"
]

BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

############################################
# DATASET
############################################

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

num_classes = len(dataset.classes)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

############################################
# TRAINING LOOP FOR EACH MODEL
############################################

for MODEL_NAME in MODELS:

    print("\nTraining", MODEL_NAME)

    model_dir = os.path.join(RESULT_DIR, MODEL_NAME)
    plot_dir = os.path.join(model_dir, "plots")
    log_dir = os.path.join(model_dir, "logs")

    os.makedirs(plot_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)

    log_path = os.path.join(log_dir, "training_log.txt")
    log_file = open(log_path, "w")

    def log(msg):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    ############################################
    # MODEL
    ############################################

    model = timm.create_model(MODEL_NAME, pretrained=True)

    for param in model.parameters():
        param.requires_grad = False

    model.reset_classifier(num_classes)

    model = model.to(DEVICE)

    optimizer = optim.Adam(
        model.get_classifier().parameters(),
        lr=LR
    )

    criterion = nn.CrossEntropyLoss()

    ############################################
    # TRAINING
    ############################################

    train_acc_list = []
    val_acc_list = []

    for epoch in range(EPOCHS):

        model.train()

        correct = 0
        total = 0

        for images, labels in tqdm(train_loader):

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        train_acc_list.append(train_acc)

        ########################################
        # VALIDATION
        ########################################

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(images)

                preds = outputs.argmax(1)

                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_acc = correct / total
        val_acc_list.append(val_acc)

        log(f"Epoch {epoch+1} TrainAcc {train_acc:.4f} ValAcc {val_acc:.4f}")

    ############################################
    # SAVE MODEL
    ############################################

    torch.save(
        model.state_dict(),
        os.path.join(model_dir, "model.pth")
    )

    ############################################
    # ACCURACY CURVE
    ############################################

    plt.figure()

    plt.plot(train_acc_list, label="Train")
    plt.plot(val_acc_list, label="Validation")

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")

    plt.legend()

    plt.title(MODEL_NAME + " Accuracy")

    plt.savefig(os.path.join(plot_dir, "accuracy_curve.png"))

    plt.close()

    ############################################
    # CONFUSION MATRIX
    ############################################

    all_preds = []
    all_labels = []

    model.eval()

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)

            outputs = model(images)

            preds = outputs.argmax(1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(14,12))

    sns.heatmap(
        cm,
        annot=False,
        xticklabels=dataset.classes,
        yticklabels=dataset.classes,
        cmap="Blues"
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.title(MODEL_NAME + " Confusion Matrix")

    plt.savefig(os.path.join(plot_dir, "confusion_matrix.png"))

    plt.close()

    ############################################
    # FEATURE EXTRACTION
    ############################################

    feature_extractor = nn.Sequential(
        *list(model.children())[:-1]
    )

    feature_extractor = feature_extractor.to(DEVICE)

    features = []
    labels_list = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)

            feats = feature_extractor(images)

            feats = feats.view(feats.size(0), -1)

            features.append(feats.cpu().numpy())
            labels_list.append(labels.numpy())

    features = np.concatenate(features)
    labels_list = np.concatenate(labels_list)

    ############################################
    # PCA
    ############################################

    pca = PCA(n_components=2)
    pca_features = pca.fit_transform(features)

    plt.figure()

    plt.scatter(
        pca_features[:,0],
        pca_features[:,1],
        c=labels_list,
        cmap="tab20",
        alpha=0.7
    )

    plt.title(MODEL_NAME + " PCA Features")

    plt.savefig(os.path.join(plot_dir, "feature_pca.png"))

    plt.close()

    ############################################
    # TSNE
    ############################################

    tsne = TSNE(n_components=2, random_state=SEED)

    tsne_features = tsne.fit_transform(features)

    plt.figure()

    plt.scatter(
        tsne_features[:,0],
        tsne_features[:,1],
        c=labels_list,
        cmap="tab20",
        alpha=0.7
    )

    plt.title(MODEL_NAME + " t-SNE Features")

    plt.savefig(os.path.join(plot_dir, "feature_tsne.png"))

    plt.close()

    log("Finished training")

    log_file.close()

print("\nAll models finished.")

In [ ]:
import os
import random
import torch
import timm
import numpy as np

from thop import profile
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

###################################
# SEED (same as training)
###################################

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

###################################
# CONFIG
###################################

DATA_DIR = "/content/train_data"
RESULT_DIR = "/content/drive/MyDrive/Linear_Probe"

MODELS = [
    "resnet50",
    "densenet121",
    "efficientnet_b0"
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

###################################
# DATASET
###################################

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

num_classes = len(dataset.classes)

###################################
# RECREATE TRAIN / VAL SPLIT
###################################

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

###################################
# EVALUATE EACH MODEL
###################################

for MODEL_NAME in MODELS:

    print("\n==========================")
    print("Model:", MODEL_NAME)

    ###################################
    # LOAD MODEL
    ###################################

    model = timm.create_model(MODEL_NAME, pretrained=False)

    model.reset_classifier(num_classes)

    model_path = os.path.join(
        RESULT_DIR,
        MODEL_NAME,
        "model.pth"
    )

    model.load_state_dict(
        torch.load(model_path, map_location=DEVICE)
    )

    model = model.to(DEVICE)
    model.eval()

    ###################################
    # PARAMETERS
    ###################################

    params = sum(p.numel() for p in model.parameters())

    print("Parameters:", params)

    ###################################
    # FLOPs and MACs
    ###################################

    dummy = torch.randn(1,3,224,224).to(DEVICE)

    macs, _ = profile(model, inputs=(dummy,), verbose=False)

    flops = macs * 2

    print("MACs:", macs)
    print("FLOPs:", flops)

    ###################################
    # VALIDATION ACCURACY
    ###################################

    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = correct / total

    print("Validation Accuracy:", acc)


        ###################################
    # CONFUSION MATRIX
    ###################################

    cm = confusion_matrix(all_labels, all_preds)

    print("Confusion Matrix Shape:", cm.shape)

    ###################################
    # PRECISION / RECALL / F1
    ###################################

    report = classification_report(
        all_labels,
        all_preds,
        target_names=dataset.classes,
        digits=4
    )

    print(report)

    ###################################
    # SAVE METRICS TO FILE
    ###################################

    metrics_path = os.path.join(
        RESULT_DIR,
        MODEL_NAME,
        "metrics.txt"
    )

    with open(metrics_path, "w") as f:

        f.write("Model: " + MODEL_NAME + "\n\n")

        f.write("Parameters: " + str(params) + "\n")
        f.write("MACs: " + str(macs) + "\n")
        f.write("FLOPs: " + str(flops) + "\n\n")

        f.write("Validation Accuracy: " + str(acc) + "\n\n")

        f.write("Precision / Recall / F1 Scores:\n\n")

        f.write(report)

    print("Metrics saved to:", metrics_path)

print("\nEvaluation complete.")

# **4.2 Fine-Tuning Strategies**
Give the path to your data directory in

DATA_DIR = "path to training dataset"

RESULT_DIR = "path to store results of finetuned strategies"

folder structure created
```bash
|- resnet50
  |-checkpoints(contains best weights in terms of val accuracy of all 4 methods)
  |-strategy.txt(logs everything)
  |-all the required pngs
|-densenet121
  |-same as resnet50
|-efficientnet_b0
```

In [ ]:
import os
import random
import torch
import timm
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from collections import defaultdict

import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split


def main():

    ########################################
    # SEED
    ########################################
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

    ########################################
    # PATHS
    ########################################
    DATA_DIR = "/content/train_data"
    RESULT_DIR = "/content/drive/MyDrive/Fine-Tuning_Strategies_final"
    os.makedirs(RESULT_DIR, exist_ok=True)

    ########################################
    # MODEL SETTINGS
    ########################################
    MODELS = ["resnet50", "densenet121", "efficientnet_b0"]

    BATCH_SIZE = 32
    # MAX_EPOCHS = 10
    MAX_EPOCHS = 3
    PATIENCE = 3
    LR = 1e-4

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    ########################################
    # DATASET
    ########################################
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
    num_classes = len(dataset.classes)

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    ########################################
    # STRATEGIES
    ########################################
    STRATEGIES = ["linear_probe","last_block","selective_20","full_finetune"]

    ########################################
    # UTILS
    ########################################
    def count_trainable(model):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        return total, trainable

    def record_trainable_layers(model, log_analysis):
        log_analysis("Trainable Layers:")
        for name, param in model.named_parameters():
            if param.requires_grad:
                log_analysis(name)

    ########################################
    # APPLY STRATEGY
    ########################################
    def apply_strategy(model, strategy):
        for p in model.parameters():
            p.requires_grad = False

        model.reset_classifier(num_classes)
        for p in model.get_classifier().parameters():
            p.requires_grad = True

        if strategy == "linear_probe":
            return
        if strategy == "last_block":
            if hasattr(model, "layer4"):
                for p in model.layer4.parameters():
                    p.requires_grad = True
        if strategy == "selective_20":
            total_params = sum(p.numel() for p in model.parameters())
            target = 0.2 * total_params
            count = 0
            for name, param in reversed(list(model.named_parameters())):
                if count < target:
                    param.requires_grad = True
                    count += param.numel()
        if strategy == "full_finetune":
            for p in model.parameters():
                p.requires_grad = True

    ########################################
    # TRAINING
    ########################################
    for MODEL_NAME in MODELS:
        print("\n======================================")
        print("Running Model:", MODEL_NAME)

        model_result_dir = os.path.join(RESULT_DIR, MODEL_NAME)
        os.makedirs(model_result_dir, exist_ok=True)

        CHECKPOINT_DIR = os.path.join(model_result_dir, "checkpoints")
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)

        analysis_path = os.path.join(model_result_dir, "strategy_analysis.txt")
        analysis_file = open(analysis_path, "w")

        def log_analysis(text):
            print(text)
            analysis_file.write(text + "\n")
            analysis_file.flush()

        for strategy in STRATEGIES:
            log_analysis("\n====================================")
            log_analysis(f"Strategy: {strategy}")

            model = timm.create_model(MODEL_NAME, pretrained=True)
            apply_strategy(model, strategy)
            model = model.to(DEVICE)

            total, trainable = count_trainable(model)
            percent = 100 * trainable / total

            log_analysis(f"Total Parameters: {total}")
            log_analysis(f"Trainable Parameters: {trainable}")
            log_analysis(f"Percentage Unfrozen: {percent:.2f}%")

            record_trainable_layers(model, log_analysis)

            optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
            criterion = nn.CrossEntropyLoss()
            scaler = GradScaler()

            train_loss_list = []
            val_acc_list = []

            best_val_acc = 0
            patience_counter = 0

            best_model_path = os.path.join(CHECKPOINT_DIR, f"{strategy}_best.pth")

            # Gradient stats grouped by layer
            grad_stats = defaultdict(list)

            #####################################
            # TRAIN LOOP
            #####################################
            for epoch in range(MAX_EPOCHS):
                model.train()
                correct = 0
                total_samples = 0
                epoch_loss = 0
                epoch_layer_grad_stats = defaultdict(list)

                for images, labels in tqdm(train_loader):
                    images = images.to(DEVICE)
                    labels = labels.to(DEVICE)

                    optimizer.zero_grad()
                    with autocast():
                        outputs = model(images)
                        loss = criterion(outputs, labels)

                    scaler.scale(loss).backward()

                    # collect gradient norms grouped by layer prefix
                    for name, param in model.named_parameters():
                        if param.grad is not None:
                            # take first 3 parts of name for grouping (works across resnet/densenet/efficientnet)
                            layer_prefix = ".".join(name.split(".")[:3])
                            grad_val = param.grad.norm().item()
                            grad_stats[layer_prefix].append(grad_val)
                            epoch_layer_grad_stats[layer_prefix].append(grad_val)

                    scaler.step(optimizer)
                    scaler.update()

                    epoch_loss += loss.item()
                    preds = outputs.argmax(1)
                    correct += (preds == labels).sum().item()
                    total_samples += labels.size(0)

                train_acc = correct / total_samples
                train_loss = epoch_loss / len(train_loader)
                train_loss_list.append(train_loss)

                #####################################
                # VALIDATION
                #####################################
                model.eval()
                correct = 0
                total_samples = 0
                with torch.no_grad():
                    for images, labels in val_loader:
                        images = images.to(DEVICE)
                        labels = labels.to(DEVICE)
                        outputs = model(images)
                        preds = outputs.argmax(1)
                        correct += (preds == labels).sum().item()
                        total_samples += labels.size(0)
                val_acc = correct / total_samples
                val_acc_list.append(val_acc)

                log_analysis(f"Epoch {epoch+1} Loss {train_loss:.4f} TrainAcc {train_acc:.4f} ValAcc {val_acc:.4f}")

                #####################################
                # GRADIENT LOGGING (per layer)
                #####################################
                log_analysis("Gradient Norm Statistics (per layer):")
                for layer, values in epoch_layer_grad_stats.items():
                    avg_grad = np.mean(values)
                    log_analysis(f"{layer} AvgGradNorm {avg_grad:.6f}")

                #####################################
                # EARLY STOPPING
                #####################################
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    patience_counter = 0
                    torch.save(model.state_dict(), best_model_path)
                    log_analysis(f"Best model updated at epoch {epoch+1}")
                else:
                    patience_counter += 1
                if patience_counter >= PATIENCE:
                    log_analysis(f"Early stopping triggered at epoch {epoch+1}")
                    break

            ########################################
            # LOSS PLOT
            ########################################
            plt.figure()
            plt.plot(train_loss_list)
            plt.xlabel("Epoch")
            plt.ylabel("Training Loss")
            plt.title(strategy + "_convergence")
            plt.savefig(os.path.join(model_result_dir, strategy + "_loss.png"))
            plt.close()

            ########################################
            # GRADIENT PLOT (per layer)
            ########################################
            layer_names = list(grad_stats.keys())
            layer_vals = [np.mean(v) for v in grad_stats.values()]

            plt.figure(figsize=(14,6))
            plt.plot(layer_names, layer_vals, marker="o")
            plt.xticks(rotation=90, fontsize=6)
            plt.xlabel("Layer")
            plt.ylabel("Average Gradient Norm")
            plt.title(f"Gradient Norm per Layer ({strategy})")

if __name__ == "__main__":
    main()

# **4.3 Few-Shot Learning Analysis**

Give the path to your data directory in

DATA_DIR = "path to training dataset"

RESULT_DIR = "path to store results of data efficiency"

folder structure created
```bash
|- resnet50
  |-checkpoints(contains best weights in terms of val accuracy of all 4 methods)
  |-experiment.txt(logs everything)
  |-all the required pngs(loss5, loss20, loss10, accuracy vs data, train_val_gap)
|-densenet121
  |-same as resnet50
|-efficientnet_b0
```

In [ ]:
import os
import random
import torch
import timm
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset

########################################
# SEED
########################################
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

########################################
# PATHS
########################################
DATA_DIR = "/content/train_data"
RESULT_DIR = "/content/drive/MyDrive/Data_Efficiency"
os.makedirs(RESULT_DIR, exist_ok=True)

########################################
# SETTINGS
########################################
MODELS = ["resnet50","densenet121","efficientnet_b0"]
DATA_FRACTIONS = [1.0,0.2,0.05]
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

########################################
# DATASET
########################################
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
num_classes = len(dataset.classes)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset,[train_size,val_size],
                                          generator=torch.Generator().manual_seed(SEED))

val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)

########################################
# EXPERIMENT LOOP
########################################
for MODEL_NAME in MODELS:
    print("\n===================================")
    print("Model:", MODEL_NAME)

    model_dir = os.path.join(RESULT_DIR, MODEL_NAME)
    os.makedirs(model_dir, exist_ok=True)

    log_file = open(os.path.join(model_dir,"experiment_log.txt"),"w")
    def log(text):
        print(text)
        log_file.write(text+"\n")
        log_file.flush()

    val_accuracies = []
    train_gaps = []

    ########################################
    # DATA FRACTION LOOP
    ########################################
    for frac in DATA_FRACTIONS:
        log("\n-----------------------------------")
        log(f"Training with {int(frac*100)}% data")

        subset_size = int(len(train_dataset)*frac)
        indices = list(range(len(train_dataset)))
        random.Random(SEED).shuffle(indices)
        subset_indices = indices[:subset_size]
        subset = Subset(train_dataset, subset_indices)

        train_loader = DataLoader(subset,batch_size=BATCH_SIZE,shuffle=True)

        ########################################
        # MODEL
        ########################################
        model = timm.create_model(MODEL_NAME,pretrained=True,num_classes=num_classes)
        model = model.to(DEVICE)

        optimizer = optim.Adam(model.parameters(),lr=LR)
        criterion = nn.CrossEntropyLoss()

        train_loss_list, train_acc_list, val_acc_list = [], [], []
        best_val_acc = 0.0
        best_model_path = os.path.join(model_dir, f"{MODEL_NAME}_{int(frac*100)}_best.pth")

        ########################################
        # TRAIN LOOP
        ########################################
        for epoch in range(EPOCHS):
            model.train()
            correct, total, loss_epoch = 0, 0, 0

            for images,labels in tqdm(train_loader):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs,labels)
                loss.backward()
                optimizer.step()

                loss_epoch += loss.item()
                preds = outputs.argmax(1)
                correct += (preds==labels).sum().item()
                total += labels.size(0)

            train_acc = correct/total
            train_loss = loss_epoch/len(train_loader)

            ########################################
            # VALIDATION
            ########################################
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for images,labels in val_loader:
                    images, labels = images.to(DEVICE), labels.to(DEVICE)
                    outputs = model(images)
                    preds = outputs.argmax(1)
                    correct += (preds==labels).sum().item()
                    total += labels.size(0)
            val_acc = correct/total

            train_loss_list.append(train_loss)
            train_acc_list.append(train_acc)
            val_acc_list.append(val_acc)

            log(f"Epoch {epoch+1} TrainLoss {train_loss:.4f} TrainAcc {train_acc:.4f} ValAcc {val_acc:.4f}")

            # Save best checkpoint
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), best_model_path)
                log(f"Best model saved at epoch {epoch+1} with ValAcc {val_acc:.4f}")

        ########################################
        # STORE RESULTS
        ########################################
        final_train = train_acc_list[-1]
        final_val = val_acc_list[-1]
        val_accuracies.append(final_val)
        gap = final_train-final_val
        train_gaps.append(gap)

        log(f"Final Validation Accuracy: {final_val:.4f}")
        log(f"Training-Validation Gap: {gap:.4f}")

        # Save final model checkpoint
        final_model_path = os.path.join(model_dir, f"{MODEL_NAME}_{int(frac*100)}_final.pth")
        torch.save(model.state_dict(), final_model_path)
        log(f"Final model saved after {EPOCHS} epochs")

        ########################################
        # LOSS CURVE
        ########################################
        plt.figure()
        plt.plot(train_loss_list)
        plt.xlabel("Epoch")
        plt.ylabel("Training Loss")
        plt.title(f"{MODEL_NAME} {int(frac*100)}% data")
        plt.savefig(os.path.join(model_dir,f"loss_{int(frac*100)}.png"))
        plt.close()

    ########################################
    # PERFORMANCE DROP
    ########################################
    acc100, acc5 = val_accuracies[0], val_accuracies[2]
    delta = (acc100-acc5)/acc100
    log("\nRelative Performance Drop:")
    log(f"Delta = {delta:.4f}")

    ########################################
    # ACCURACY VS DATA
    ########################################
    perc = [100,20,5]
    plt.figure()
    plt.plot(perc,val_accuracies,marker="o")
    plt.xlabel("Training Data (%)")
    plt.ylabel("Validation Accuracy")
    plt.title(f"{MODEL_NAME} Data Efficiency")
    plt.savefig(os.path.join(model_dir,"accuracy_vs_data.png"))
    plt.close()

    ########################################
    # OVERFITTING GAP
    ########################################
    plt.figure()
    plt.plot(perc,train_gaps,marker="o")
    plt.xlabel("Training Data (%)")
    plt.ylabel("Train-Val Gap")
    plt.title(f"{MODEL_NAME} Overfitting")
    plt.savefig(os.path.join(model_dir,"train_val_gap.png"))
    plt.close()

    log_file.close()

print("\nExperiment Finished")


# **4.4 Corruption Robustness Evaluation**

Give the path to your data directory in

DATA_DIR = "path to training dataset"

MODEL_DIR = "path for finetuned weights folder generated in 4.2"

RESULT_DIR = "path to store Robustness resultss"

folder structure created
```bash
|- resnet50
  |-noise_accuracy.png(validation accuracy vs data trained)
  |-log.txt(contains all the required calculations)
|-densenet121
  |-same as resnet50
|-efficientnet_b0
```

In [ ]:
import os
import torch
import timm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from PIL import Image, ImageFilter
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

############################################
# PATHS
############################################

DATA_DIR = "/content/train_data"
MODEL_DIR = "/content/drive/MyDrive/Fine-Tuning_Strategies"
RESULT_DIR = "/content/drive/MyDrive/Robustness_Results"

os.makedirs(RESULT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

############################################
# MODELS
############################################

MODELS = [
    "resnet50",
    "densenet121",
    "efficientnet_b0"
]

############################################
# CORRUPTION SETTINGS
############################################

NOISE_LEVELS = [0.05, 0.1, 0.2]
BRIGHTNESS_LEVELS = [0.2, 0.4]

############################################
# BASE TRANSFORM
############################################

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

num_classes = len(dataset.classes)

loader = DataLoader(dataset, batch_size=32, shuffle=False)

############################################
# CORRUPTION FUNCTIONS
############################################

def gaussian_noise(img, sigma):

    noise = torch.randn_like(img)*sigma
    img = img + noise
    img = torch.clamp(img,0,1)

    return img

def motion_blur(img):

    img = transforms.ToPILImage()(img)
    img = img.filter(ImageFilter.GaussianBlur(radius=3))
    img = transforms.ToTensor()(img)

    return img

def brightness_shift(img, factor):

    img = transforms.ToPILImage()(img)
    img = transforms.functional.adjust_brightness(img,1+factor)
    img = transforms.ToTensor()(img)

    return img

############################################
# EVALUATION FUNCTION
############################################

def evaluate(model, corruption=None):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(DEVICE)

            if corruption is not None:

                images = torch.stack([
                    corruption(img.cpu()).to(DEVICE)
                    for img in images
                ])

            labels = labels.to(DEVICE)

            outputs = model(images)

            preds = outputs.argmax(1)

            correct += (preds==labels).sum().item()
            total += labels.size(0)

    return correct/total

############################################
# MAIN EXPERIMENT
############################################

results = []

for MODEL_NAME in MODELS:

    print("\n===================================")
    print("Model:", MODEL_NAME)

    model_result_dir = os.path.join(RESULT_DIR, MODEL_NAME)
    os.makedirs(model_result_dir, exist_ok=True)

    log_path = os.path.join(model_result_dir,"log.txt")
    log_file = open(log_path,"w")

    def log(text):
        print(text)
        log_file.write(text+"\n")
        log_file.flush()

    ########################################
    # LOAD MODEL
    ########################################

    model = timm.create_model(
        MODEL_NAME,
        pretrained=False,
        num_classes=num_classes
    )

    checkpoint = os.path.join(
        MODEL_DIR,
        MODEL_NAME,
        "checkpoints",
        "full_finetune_best.pth"
    )

    model.load_state_dict(torch.load(checkpoint,map_location=DEVICE))

    model = model.to(DEVICE)

    ########################################
    # CLEAN ACCURACY
    ########################################

    clean_acc = evaluate(model)

    log(f"Clean Accuracy: {clean_acc:.4f}")

    ########################################
    # GAUSSIAN NOISE
    ########################################

    noise_accs = []

    for sigma in NOISE_LEVELS:

        acc = evaluate(
            model,
            corruption=lambda x: gaussian_noise(x,sigma)
        )

        error = 1-acc
        robustness = acc/clean_acc

        noise_accs.append(acc)

        log(
            f"Gaussian Noise σ={sigma} "
            f"Accuracy={acc:.4f} "
            f"CorruptionError={error:.4f} "
            f"RelativeRobustness={robustness:.4f}"
        )

        results.append(
            [MODEL_NAME,"noise",sigma,acc,error,robustness]
        )

    ########################################
    # MOTION BLUR
    ########################################

    blur_acc = evaluate(model,motion_blur)

    error = 1-blur_acc
    robustness = blur_acc/clean_acc

    log(
        f"MotionBlur Accuracy={blur_acc:.4f} "
        f"Error={error:.4f} "
        f"Robustness={robustness:.4f}"
    )

    results.append(
        [MODEL_NAME,"motion_blur",0,blur_acc,error,robustness]
    )

    ########################################
    # BRIGHTNESS SHIFT
    ########################################

    for b in BRIGHTNESS_LEVELS:

        acc = evaluate(
            model,
            corruption=lambda x: brightness_shift(x,b)
        )

        error = 1-acc
        robustness = acc/clean_acc

        log(
            f"BrightnessShift +{b} "
            f"Accuracy={acc:.4f} "
            f"Error={error:.4f} "
            f"Robustness={robustness:.4f}"
        )

        results.append(
            [MODEL_NAME,"brightness",b,acc,error,robustness]
        )

    ########################################
    # NOISE PLOT
    ########################################

    plt.figure()

    plt.plot(NOISE_LEVELS,noise_accs,marker="o")

    plt.xlabel("Noise σ")
    plt.ylabel("Accuracy")

    plt.title(f"{MODEL_NAME} Noise Robustness")

    plt.savefig(os.path.join(model_result_dir,"noise_accuracy.png"))

    plt.close()

    log_file.close()

############################################
# SAVE GLOBAL RESULTS
############################################

df = pd.DataFrame(
    results,
    columns=[
        "model",
        "corruption",
        "level",
        "accuracy",
        "corruption_error",
        "relative_robustness"
    ]
)

csv_path = os.path.join(RESULT_DIR,"robustness_results.csv")

df.to_csv(csv_path,index=False)

############################################
# GLOBAL COMPARISON PLOTS
############################################

for corruption in df["corruption"].unique():

    subset = df[df["corruption"]==corruption]

    plt.figure()

    for model in MODELS:

        model_data = subset[subset["model"]==model]

        plt.plot(
            model_data["level"],
            model_data["accuracy"],
            marker="o",
            label=model
        )

    plt.xlabel("Corruption Level")
    plt.ylabel("Accuracy")

    plt.title(f"Model Robustness: {corruption}")

    plt.legend()

    plt.savefig(
        os.path.join(
            RESULT_DIR,
            f"{corruption}_comparison.png"
        )
    )

    plt.close()

print("\nRobustness evaluation complete.")

# **4.5 Layer-Wise Feature Probing**

DATA_DIR = "path to training dataset"

MODEL_DIR = "path to finetuning strategies folder from part 4.2"

RESULT_DIR = "path to folder where results are to be stored"

```bash
Representation_Analysis/
│
├── resnet50/
│   ├── log.txt
│   ├── accuracy_vs_depth.png
│   ├── pca_early.png
│   ├── pca_middle.png
│   ├── pca_final.png
│
├── densenet121/
│   ├── same as resnet50
├── efficientnet_b0/
```

In [ ]:
import os, gc
import torch, timm, random
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import IncrementalPCA
from sklearn.linear_model import LogisticRegression
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

############################################
# SETTINGS
############################################

DATA_DIR = "/content/train_data"
MODEL_DIR = "/content/drive/MyDrive/Fine-Tuning_Strategies"
RESULT_DIR = "/content/drive/MyDrive/Representation_Analysis"

os.makedirs(RESULT_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = ["resnet50","densenet121","efficientnet_b0"]

############################################
# SEED
############################################

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

############################################
# DATASET
############################################

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
num_classes = len(dataset.classes)

# train / validation split
val_ratio = 0.2
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset,batch_size=64,shuffle=False)
val_loader   = DataLoader(val_dataset,batch_size=64,shuffle=False)

############################################
# PCA SUBSET (30 samples per class)
############################################

subset_indices, class_counts = [], {}

for i,(img,label) in enumerate(dataset):
    if label not in class_counts:
        class_counts[label] = 0
    if class_counts[label] < 30:
        subset_indices.append(i)
        class_counts[label]+=1
    if len(class_counts)==num_classes and all(v>=30 for v in class_counts.values()):
        break

subset = torch.utils.data.Subset(dataset,subset_indices)
subset_loader = DataLoader(subset,batch_size=64)

############################################
# FEATURE EXTRACTION
############################################

def extract_features(model, layer_name, loader):

    layer = dict([*model.named_modules()])[layer_name]

    feats = []
    labels = []

    def hook(module,input,output):
        pooled = torch.nn.functional.adaptive_avg_pool2d(output,(1,1))
        feats.append(pooled.view(pooled.size(0),-1).cpu())

    handle = layer.register_forward_hook(hook)

    model.eval()

    with torch.no_grad():
        for images,lab in loader:
            images = images.to(DEVICE)
            model(images)
            labels.extend(lab.numpy())

    handle.remove()

    X = torch.cat(feats).numpy()
    y = np.array(labels)

    feats.clear()
    gc.collect()
    torch.cuda.empty_cache()

    return X,y

############################################
# MAIN EXPERIMENT
############################################

for MODEL_NAME in MODELS:

    print("\n===============================")
    print("Model:",MODEL_NAME)

    model_dir = os.path.join(RESULT_DIR,MODEL_NAME)
    os.makedirs(model_dir,exist_ok=True)

    log_file = open(os.path.join(model_dir,"log.txt"),"w")

    def log(x):
        print(x)
        log_file.write(x+"\n")
        log_file.flush()

    ########################################
    # LOAD MODEL
    ########################################

    model = timm.create_model(
        MODEL_NAME,
        pretrained=False,
        num_classes=num_classes
    )

    checkpoint = os.path.join(
        MODEL_DIR,
        MODEL_NAME,
        "checkpoints",
        "full_finetune_best.pth"
    )

    model.load_state_dict(torch.load(checkpoint,map_location=DEVICE))
    model = model.to(DEVICE)

    ########################################
    # SELECT LAYERS
    ########################################

    if MODEL_NAME=="resnet50":
        layers = {"early":"layer1","middle":"layer3","final":"layer4"}

    elif MODEL_NAME=="densenet121":
        layers = {
            "early":"features.denseblock1",
            "middle":"features.denseblock3",
            "final":"features.denseblock4"
        }

    else:
        layers = {"early":"blocks.1","middle":"blocks.4","final":"blocks.6"}

    depth_acc = []

    ########################################
    # ANALYSIS PER LAYER
    ########################################

    for depth,layer in layers.items():

        log(f"\nLayer {depth} → {layer}")

        # Extract features once
        X_train,y_train = extract_features(model,layer,train_loader)
        X_val,y_val     = extract_features(model,layer,val_loader)

        ####################################
        # Linear Probe
        ####################################

        clf = LogisticRegression(max_iter=1000,n_jobs=-1)

        clf.fit(X_train,y_train)
        acc = clf.score(X_val,y_val)

        depth_acc.append(acc)

        log(f"Validation Accuracy: {acc}")

        ####################################
        # Feature Norm Statistics
        ####################################

        norms = np.linalg.norm(X_val,axis=1)

        log(f"Feature Norm Mean: {norms.mean()}")
        log(f"Feature Norm Std: {norms.std()}")

        ####################################
        # PCA
        ####################################

        X_subset,y_subset = extract_features(model,layer,subset_loader)

        pca = IncrementalPCA(n_components=2)

        X2 = pca.fit_transform(X_subset)

        plt.figure(figsize=(8,6))

        plt.scatter(X2[:,0],X2[:,1],c=y_subset,cmap="tab20",s=10)

        plt.title(f"{MODEL_NAME} PCA {depth}")

        plt.savefig(os.path.join(model_dir,f"pca_{depth}.png"))
        plt.close()

        del X_train,y_train,X_val,y_val,X_subset,y_subset,X2,pca,clf
        gc.collect()
        torch.cuda.empty_cache()

    ########################################
    # ACCURACY VS DEPTH
    ########################################

    plt.figure()

    plt.plot(["early","middle","final"],depth_acc,marker="o")

    plt.ylabel("Validation Accuracy")
    plt.title(f"{MODEL_NAME} Depth vs Accuracy")

    plt.savefig(os.path.join(model_dir,"accuracy_vs_depth.png"))
    plt.close()

    log_file.close()

print("\nRepresentation analysis complete")